In [ ]:
ORIGIN = "Portland, OR"
MILES_PER_DAY = 500
RESOLUTION = 5

## Step 1 - Innitialize Database containing all points, lines and geometries in the United States 

* Can function simmilar to Google Maps but runs locally/offline 
* 10s of millions of datapoints (148 GB)
* "PostGIS" Database (PostgreSQL + Geographic Information Systems)

<div style="display:flex; gap:5px;">
  <img src="photos/osm_data_example_points.png" style="max-width:50%;">
  <img src="photos/osm_data_example_geometries_lines.png" style="max-width:50%;">
</div>

## Step 2 - Split the US into many even size areas (and find a road point within as many areas as possible)

Only 3 Polygons tile regularly: Triangles, Hexagons & Squares
- no gaps
- no overlaps
- identical orientation at every vertex (same angle pattern everywhere)

| Triangle ❌ | Square ❌ | Hexagon ✅ |
|----------|---------|----------|
| ![](photos/neighbors-triangle.png) | ![](photos/neighbors-square.png) | ![](photos/neighbors-hexagon.png) |
| Triangles have 12 neighbors | Squares have 8 neighbors | Hexagons have 6 *equidistant* neighbors |

Hexagons allow for the simpliest analysis of 2D movement (and look the best)

In [ ]:
import h3
from h3 import LatLngPoly, LatLngMultiPoly
import json
from shapely.geometry import shape, Polygon, MultiPolygon, mapping
from shapely.ops import unary_union
from sqlalchemy import create_engine, text
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
import folium
import numpy as np
# import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import colormaps

resolution = 5
ENGINE = create_engine(
    "postgresql+psycopg2://nfleck:@localhost:5432/conus_osm_routing"
)

def draw_conus_polygon():
    with open("datasets/conus-states.json", "r") as f:
        state_geo = json.load(f)

    states = [shape(f["geometry"]) for f in state_geo["features"]]
    conus = unary_union(states)
    return conus

def convert_polygon_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_polygon_to_h3(p) for p in geom.geoms))

def initialize_cell_centers_dataframe(cells):
    rows = [
        (cell, *h3.cell_to_latlng(cell))
        for cell in cells
    ]

    df = pd.DataFrame(rows, columns=["CellID", "Latitude", "Longitude"])

    df["Latitude"] = df["Latitude"].round(6)
    df["Longitude"] = df["Longitude"].round(6)
    return df

def get_road_snapped_point(lon, lat):
    
    sql = text("""
        SELECT
          ST_Y(geom_4326) AS lat,
          ST_X(geom_4326) AS lon
        FROM (
          SELECT
            ST_Transform(
              ST_ClosestPoint(way, input.geom),
              4326
            ) AS geom_4326
          FROM planet_osm_roads
          CROSS JOIN (
            SELECT ST_Transform(
                    ST_SetSRID(ST_Point(:lon, :lat), 4326),
                    3857
                  ) AS geom
          ) AS input
          WHERE highway IS NOT NULL
          ORDER BY way <-> input.geom
          LIMIT 1
        ) snapped;
    """) 
    
    with ENGINE.connect() as conn:
      result = conn.execute(sql, {"lon": lon, "lat": lat}).fetchone()

    rslat = round(result[0], 6)
    rslon = round(result[1], 6)

    return rslat, rslon

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return round(R * c)

def add_road_snapped_data_to_dataframe(df):
    df[["RSLatitude", "RSLongitude"]] = df.apply(
        lambda row: get_road_snapped_point(row["Longitude"], row["Latitude"]), 
        axis=1, 
        result_type="expand"
    )

    df["RSDistance"] = df.apply(
        lambda row: haversine(
            row["Latitude"],
            row["Longitude"],
            row["RSLatitude"],
            row["RSLongitude"]
        ),
        axis=1
    )

    # check if rs lat/lon is in same cellID
    df["RSCellID"] = df.apply(
        lambda row : h3.latlng_to_cell(
            row["RSLatitude"], 
            row["RSLongitude"], 
            resolution
            ),
            axis=1
    )

    df["ValidSnap"] = df["CellID"] == df["RSCellID"]   
    return df    

conus = draw_conus_polygon()
h3_conus = convert_polygon_to_h3(conus)
h3_cells = h3.polygon_to_cells_experimental(
    h3shape=h3_conus, 
    res=resolution, 
    contain="overlap"
)
df = initialize_cell_centers_dataframe(h3_cells)
df = add_road_snapped_data_to_dataframe(df)

def draw_conus_polygon():
    with open("datasets/conus-states.json", "r") as f:
        state_geo = json.load(f)

    states = [shape(f["geometry"]) for f in state_geo["features"]]
    conus = unary_union(states)
    return conus

def convert_polygon_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_polygon_to_h3(p) for p in geom.geoms))

def initialize_cell_centers_dataframe(cells):
    rows = [
        (cell, *h3.cell_to_latlng(cell))
        for cell in cells
    ]

    df = pd.DataFrame(rows, columns=["CellID", "Latitude", "Longitude"])

    df["Latitude"] = df["Latitude"].round(6)
    df["Longitude"] = df["Longitude"].round(6)
    return df

def get_road_snapped_point(lon, lat):
    
    sql = text("""
        SELECT
          ST_Y(geom_4326) AS lat,
          ST_X(geom_4326) AS lon
        FROM (
          SELECT
            ST_Transform(
              ST_ClosestPoint(way, input.geom),
              4326
            ) AS geom_4326
          FROM planet_osm_roads
          CROSS JOIN (
            SELECT ST_Transform(
                    ST_SetSRID(ST_Point(:lon, :lat), 4326),
                    3857
                  ) AS geom
          ) AS input
          WHERE highway IS NOT NULL
          ORDER BY way <-> input.geom
          LIMIT 1
        ) snapped;
    """) 
    
    with ENGINE.connect() as conn:
      result = conn.execute(sql, {"lon": lon, "lat": lat}).fetchone()

    rslat = round(result[0], 6)
    rslon = round(result[1], 6)

    return rslat, rslon

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return round(R * c)

def add_road_snapped_data_to_dataframe(df):
    df[["RSLatitude", "RSLongitude"]] = df.apply(
        lambda row: get_road_snapped_point(row["Longitude"], row["Latitude"]), 
        axis=1, 
        result_type="expand"
    )

    df["RSDistance"] = df.apply(
        lambda row: haversine(
            row["Latitude"],
            row["Longitude"],
            row["RSLatitude"],
            row["RSLongitude"]
        ),
        axis=1
    )

    # check if rs lat/lon is in same cellID
    df["RSCellID"] = df.apply(
        lambda row : h3.latlng_to_cell(
            row["RSLatitude"], 
            row["RSLongitude"], 
            resolution
            ),
            axis=1
    )

    df["ValidSnap"] = df["CellID"] == df["RSCellID"]   
    return df    

conus = draw_conus_polygon()
h3_conus = convert_polygon_to_h3(conus)
h3_cells = h3.polygon_to_cells_experimental(
    h3shape=h3_conus, 
    res=resolution, 
    contain="overlap"
)
df = initialize_cell_centers_dataframe(h3_cells)
df = add_road_snapped_data_to_dataframe(df)

df = df.loc[df["ValidSnap"] == True]

m = folium.Map(location=(40, -100), zoom_start=5)

for index, row in df.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=0.25,
        color="red"
    ).add_to(m)

    folium.CircleMarker(
        location=[row["RSLatitude"], row["RSLongitude"]],
        radius=0.334,
        color="green"
    ).add_to(m)


    folium.PolyLine(
        locations=[
            [row["Latitude"], row["Longitude"]],
            [row["RSLatitude"], row["RSLongitude"]]
            ],
        color="green",
        weight=0.5
    ).add_to(m)

    hexagon = h3.cell_to_boundary(row["CellID"])

    folium.Polygon(
        locations=hexagon,
        weight=0.5,
        fill=False
    ).add_to(m)
m


## Step 3 - calculate isochrone areas

Once we select a specific hexagon as the starting point, how do we determine which other hexagons are within 1 day transit, 2 day transit, etc.

### Methods for Calulating Isochrone areas (Methods for navigating green dots)

| Method 1 - Neighbor to neighbor traversal (green dot to neighboring green dots) | Method 2 - Full Origin/Destination pairing table | Method 3 - On the fly transit calculation for single-origin/many-destination |
|---|---|---|
|Green dot to neighboring green dots <ul><li>Fast generation</li><li>Relativly small intermediary data table</li><li>Complex algorithms needed to find ideal routes and translate to isochrones</li></ul>| Every green dot to everyother green dot <ul><li>Fast generation (single column or row look up)</li><li>Massive data requirements, largest intermediary data table <li>limited to low resolution</li>$$T(r) \sim 1.6 \cdot 49^{r}$$</ul> | <ul><li>No intermediary data table</li><li>Slower generation</li><li>Simple implementation</li></ul> |


In [ ]:

# -----------------------
# Config
# -----------------------

GRADIENT_ISOCHRONE = False

ENGINE = create_engine(
    "postgresql+psycopg2://nfleck:@localhost:5432/conus_osm_routing"
)

MAIN_QUERY_BATCH_SIZE = 5940

ISOCHRONE_COLOR_SCHEME = "Spectral_r"
# ISOCHRONE_COLOR_SCHEME = "Spectral"
# ISOCHRONE_COLOR_SCHEME = "HSV"
# ISOCHRONE_COLOR_SCHEME = "RdYlGn_r"


miles_adjustment_ratio = 1.15 # makes osm transit closer to google maps (probably based on sample test)
MILES_PER_DAY = MILES_PER_DAY * miles_adjustment_ratio

# -----------------------
# Load and prep data
# -----------------------

def get_origin_coordinates(city_state):
    state = city_state[-2:]
    city = city_state[:-4]

    origin_df = pd.read_csv("datasets/city_lat_long.csv")

    origin_df = origin_df.loc[
        (origin_df["city"] == city) &
        (origin_df["state"] == state)].reset_index(drop=True)

    START_LAT = origin_df.loc[0, "latitude"] 
    START_LNG = origin_df.loc[0, "longitude"]
    return START_LAT, START_LNG

START_LAT, START_LNG = get_origin_coordinates(ORIGIN)

start_cell = h3.latlng_to_cell(
    lat=START_LAT,
    lng=START_LNG,
    res=RESOLUTION
)

df = pd.read_csv(f"datasets/initial_res_{RESOLUTION}_points.csv")
df = df.loc[df["ValidSnap"] == True]

df = df.drop(
    columns=[
        "neighbors",
        "ValidNeighbors",
        "ValidSnap",
        "RSCellID",
        "RSDistance",
    ]
)

origin_rs_lat = float(df.loc[df["CellID"] == start_cell, "RSLatitude"].iloc[0])
origin_rs_lng = float(df.loc[df["CellID"] == start_cell, "RSLongitude"].iloc[0])

# -----------------------
# Prepare destination dataframe
# -----------------------

dest_df = df[["CellID", "RSLatitude", "RSLongitude"]].rename(
    columns={
        "CellID": "cell_id",
        "RSLatitude": "lat",
        "RSLongitude": "lng",
    }
)

# -----------------------
# Execute routing
# -----------------------

with ENGINE.begin() as conn:

    # 1️⃣ Create temp table for raw destination points
    conn.execute(text("""
        CREATE TEMP TABLE destination_points (
            cell_id TEXT,
            lat DOUBLE PRECISION,
            lng DOUBLE PRECISION
        ) ON COMMIT DROP;
    """))

    dest_df.to_sql(
        "destination_points",
        conn,
        if_exists="append",
        index=False,
        method="multi"
    )

    # 2️⃣ Nearest routing vertex for each destination
    conn.execute(text("""
        CREATE TEMP TABLE destinations AS
        SELECT
            d.cell_id,
            v.id AS node_id
        FROM destination_points d
        JOIN LATERAL (
            SELECT id
            FROM routing_vertices v
            ORDER BY v.geom <-> ST_Transform(
                ST_SetSRID(ST_Point(d.lng, d.lat), 4326),
                3857
            )
            LIMIT 1
        ) v ON true;
    """))

    conn.execute(text("CREATE INDEX ON destinations (node_id);"))

    # 3️⃣ Fetch distinct node_ids
    node_ids = pd.read_sql(
        "SELECT DISTINCT node_id FROM destinations",
        conn
    )["node_id"].tolist()

    print(f"Finding transit distances: {ORIGIN} to {len(node_ids)} unique destinations")

    # 4️⃣ Batched Dijkstra over routing_edges
    batch_sql = text("""
    WITH origin AS (
        SELECT v.id AS id
        FROM routing_vertices v
        ORDER BY v.geom <-> ST_Transform(
            ST_SetSRID(ST_Point(:lng1, :lat1), 4326),
            3857
        )
        LIMIT 1
    ),
    routes AS (
        SELECT *
        FROM pgr_dijkstra(
            $$SELECT id,
                    source,
                    target,
                    length_m AS cost,
                    length_m AS reverse_cost
            FROM routing_edges
            WHERE source IS NOT NULL
                AND target IS NOT NULL$$,
            (SELECT id FROM origin),
            :node_array,
            true
        )
    )
    SELECT
        end_vid AS node_id,
        MAX(agg_cost) AS driving_distance_m
    FROM routes
    GROUP BY end_vid;
    """)

    # 5️⃣ Run in batches
    all_results = []

    for i in range(0, len(node_ids), MAIN_QUERY_BATCH_SIZE):
        batch = node_ids[i:i + MAIN_QUERY_BATCH_SIZE]

        print(
            f"Batch {i // MAIN_QUERY_BATCH_SIZE + 1} "
            f"({(i // MAIN_QUERY_BATCH_SIZE + 1) * len(batch)}/{len(node_ids)} Destinations)"
        )


        result = pd.read_sql(
            batch_sql,
            conn,
            params={
                "lat1": origin_rs_lat,
                "lng1": origin_rs_lng,
                "node_array": batch
            }
        )

        all_results.append(result)

    result_df = pd.concat(all_results, ignore_index=True)

    # 6️⃣ Get cell_id → node_id mapping
    dest_map = pd.read_sql(
        "SELECT cell_id, node_id FROM destinations",
        conn
    )

# -----------------------
# Merge results back
# -----------------------

node_dist_map = result_df.set_index("node_id")["driving_distance_m"]

dest_map["driving_distance_m"] = dest_map["node_id"].map(node_dist_map)

df = df.merge(
    dest_map[["cell_id", "driving_distance_m"]],
    left_on="CellID",
    right_on="cell_id",
    how="left"
).drop(columns=["cell_id"])

#####################################################################
### Populate missing cells using average of surrounding neighbors ###
#####################################################################

print("Cleaning missing cells based on neighbors")

df["avg_neighb_dist"] = pd.Series(dtype="float64")

# loop
# step 1 find number and list of valid neighbors for each cell where driving distance needs to be found 
def get_non_null_cells(df, series):
    non_null_cells = set(df.loc[series.notnull(), "CellID"])
    return non_null_cells

# step 2 for those cells needing driving distance, find those with the max number of valid neighbors
def get_neighbors(h3_index, k=1):
    neighbors = h3.grid_disk(h3_index, k)
    neighbors.remove(h3_index)
    return list(neighbors)

def count_valid_neighbors(neighbors, non_nulls):
    return sum(n in non_nulls for n in neighbors)

def list_valid_neighbors(neighbors, non_nulls):
    return list(set(neighbors) & non_nulls)

def find_max_neighbors(df):
    df = df.loc[df["driving_distance_m"].isnull()]
    max_neighbors = df["valid_neighbor_count"].max()
    return max_neighbors

# step 3 set driving distance to average of valid neighboring cells
def avg_neighbor_transit(valid_neighbors):
    if not valid_neighbors:
        return np.nan     # instead of None
    
    neighbor_dists = df.loc[
        df["CellID"].isin(valid_neighbors),
        "driving_distance_m"
    ].dropna()
    
    if len(neighbor_dists) == 0:
        return np.nan     # instead of None
    
    return neighbor_dists.mean()

# step 4 repeat until all driving distances are populated
def update_avgs():
    global df
    
    non_null_cells = get_non_null_cells(df, df["driving_distance_m"])
    
    df["neighbors"] = df["CellID"].apply(get_neighbors)

    df["valid_neighbor_count"] = df["neighbors"].apply(
        lambda x: count_valid_neighbors(x, non_null_cells)
    )
    
    df["valid_neighbors"] = df["neighbors"].apply(
        lambda x: list_valid_neighbors(x, non_null_cells)
    )
    
    max_neighbs = find_max_neighbors(df)
    
    # Select rows that:
    # 1. Need filling
    # 2. Have the maximum number of valid neighbors
    mask = (
        df["driving_distance_m"].isnull() &
        (df["valid_neighbor_count"] == max_neighbs)
    )
    
    df.loc[mask, "avg_neighb_dist"] = df.loc[mask, "valid_neighbors"].apply(
        avg_neighbor_transit
    )
    
    # Now actually fill the missing driving distances
    df.loc[mask, "driving_distance_m"] = df.loc[mask, "avg_neighb_dist"]

# while there are null values in driving_distance_m, loop update_avgs()
while df["driving_distance_m"].isnull().any():
    prev_nulls = df["driving_distance_m"].isnull().sum()
    
    update_avgs()
    
    new_nulls = df["driving_distance_m"].isnull().sum()
    
    if new_nulls == prev_nulls:
        # print("No further updates possible. Stopping.")
        break

##############################
### Set Isochrone Geometry ###
##############################

print ("Setting Isochrone Geometries")

df["driving_distance_miles"] = df["driving_distance_m"]/1608.344

df["driving_distance_days"] = (
    pd.to_numeric(df["driving_distance_miles"], errors="coerce")
    / MILES_PER_DAY
).apply(np.ceil)

df = df.dropna(subset=["driving_distance_days"])

unique_transit_days = df["driving_distance_days"].unique()

def get_unique_day_polygon(df, transit_day):
    filtered_df = df.loc[df["driving_distance_days"] == transit_day]
    h3_indexes = filtered_df["CellID"].unique()

    polygons = []

    for h in h3_indexes:
        boundary = h3.cell_to_boundary(h)
        polygons.append(Polygon(boundary))

    merged = unary_union(polygons)

    if isinstance(merged, Polygon):
        merged = MultiPolygon([merged])

    return merged

day_polygons = {
    day: get_unique_day_polygon(df, day)
    for day in unique_transit_days
}

In [ ]:
###############################
### Generate & populate map ###
###############################

print("Populating map")

def flip_coords(geom):
    """Swap (x, y) → (y, x) recursively for Polygon/MultiPolygon"""
    if geom.is_empty:
        return geom

    geom = geom.buffer(0.001).buffer(-0.001) # remove multipolygon internal artifacts

    if geom.geom_type == "Polygon":
        exterior = [(y, x) for x, y in geom.exterior.coords]
        interiors = [[(y, x) for x, y in i.coords] for i in geom.interiors]
        return Polygon(exterior, interiors)
    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([flip_coords(p) for p in geom.geoms])
    else:
        return geom

# Choose a colormap from matplotlib
cmap = colormaps["Spectral_r"]

max_transit_miles = df["driving_distance_miles"].max()

norm_miles = mcolors.Normalize(vmin=0, vmax=max_transit_miles)

m = folium.Map(
    location=(40, -100), 
    zoom_start=5, 
    tiles= "Cartodb dark_matter"
    )

if GRADIENT_ISOCHRONE:
    for index, row in df.iterrows():
        hexagon = h3.cell_to_boundary(row["CellID"])
        value = row["driving_distance_miles"]

        # Map value to a color
        hex_fill_color = mcolors.to_hex(cmap(norm_miles(value)))

        folium.Polygon(
            locations=hexagon,
            weight=0.25,
            color=None,
            fill=True,
            fill_color=hex_fill_color,
            fill_opacity=0.2
        ).add_to(m)

norm_days = mcolors.Normalize(
    vmin=unique_transit_days.min(),
    vmax=unique_transit_days.max()
)

for day, polygon in day_polygons.items():
    polygon = flip_coords(polygon)  # <-- swap coords for Folium

    isochrone_color = mcolors.to_hex(cmap(norm_days(day)))
    folium.GeoJson(
        data={
            "type": "Feature",
            "geometry": mapping(polygon),
            "properties": {}
        },
        style_function=lambda feature, col=isochrone_color: {
            "fill": not GRADIENT_ISOCHRONE,
            "color": col,
            "opacity": 1,
            "weight": 1
        }
    ).add_to(m)

folium.CircleMarker(
    location=[START_LAT, START_LNG],
    radius=5,
    color="#AFE722",
    weight=1,
    fill=True,
    fill_color="green",
    fill_opacity=0.7,
    popup=ORIGIN
).add_to(m)

m

## TO DO
- more functions - 1 main generate_isochrone() function
- switch pandas to polars
- optimize main transit distance query
- make variable and column names consistent format
- output in floating window rather than in notebook
- organize project folder

Optimize isochrone generation with compaction, only use high resolution on edges of isochrone layers
| Uncompaced (dense) | Compacted (Sparse) |
|--------------------|--------------------|
| All high resolution - 10633 cells | High resolution only where necessary - 901 cells|
|![](photos/ca_uncompact.png)|![](photos/ca_compact.png)|